ETAPA 1 — Modelos de tradução

1.1 RNN Tradutor

# RNN/LSTM

RNN é uma rede neural com memória, usada quando a ordem dos dados importa.

A LSTM é uma evolução da RNN. Ela foi criada para lidar melhor com dependências longas. Otimiza memória, tenta resolver a pergunta: “o que eu devo manter na memória e o que posso descartar?”.



In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:


# Base pequena de exemplo
entrada_textos = [
    "hello",
    "good morning",
    "good night",
    "how are you",
    "thank you",
    "i love you",
    "see you later"
]

saida_textos = [
    "<start> olá <end>",
    "<start> bom dia <end>",
    "<start> boa noite <end>",
    "<start> como você está <end>",
    "<start> obrigado <end>",
    "<start> eu te amo <end>",
    "<start> até mais tarde <end>"
]

In [ ]:
# Tokenização
tokenizer_entrada = Tokenizer()
tokenizer_saida = Tokenizer(filters='')

tokenizer_entrada.fit_on_texts(entrada_textos)
tokenizer_saida.fit_on_texts(saida_textos)

entrada_seq = tokenizer_entrada.texts_to_sequences(entrada_textos)
saida_seq = tokenizer_saida.texts_to_sequences(saida_textos)

max_entrada = max(len(seq) for seq in entrada_seq)
max_saida = max(len(seq) for seq in saida_seq)

entrada_seq = pad_sequences(entrada_seq, maxlen=max_entrada, padding="post")
saida_seq = pad_sequences(saida_seq, maxlen=max_saida, padding="post")

vocab_entrada = len(tokenizer_entrada.word_index) + 1
vocab_saida = len(tokenizer_saida.word_index) + 1

# Entrada e saída do decoder
decoder_input = saida_seq[:, :-1]
decoder_output = saida_seq[:, 1:]

In [ ]:
# Modelo Encoder-Decoder com LSTM
latent_dim = 64

# Configuração do Encoder
encoder_inputs = Input(shape=(max_entrada,))
encoder_emb = Embedding(vocab_entrada, latent_dim)(encoder_inputs)
encoder_lstm, state_h, state_c = LSTM(latent_dim, return_state=True)(encoder_emb)

# Configuração do Decoder
decoder_inputs = Input(shape=(max_saida - 1,))
decoder_emb = Embedding(vocab_saida, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True)(decoder_emb, initial_state=[state_h, state_c])
decoder_dense = Dense(vocab_saida, activation="softmax")(decoder_lstm)

# Definição e Compilação do Modelo
model = Model([encoder_inputs, decoder_inputs], decoder_dense)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Treinamento
model.fit(
    [entrada_seq, decoder_input],
    np.expand_dims(decoder_output, -1),
    epochs=300,
    verbose=0
)

print("Modelo RNN/LSTM treinado.")

Modelo RNN/LSTM treinado.


In [ ]:
def traduzir(frase):
    entrada = tokenizer_entrada.texts_to_sequences([frase])
    entrada = pad_sequences(entrada, maxlen=max_entrada, padding="post")

    decoder_seq = tokenizer_saida.texts_to_sequences(["<start>"])[0]
    decoder_seq = pad_sequences([decoder_seq], maxlen=max_saida - 1, padding="post")

    pred = model.predict([entrada, decoder_seq], verbose=0)
    pred_indices = np.argmax(pred[0], axis=1)

    palavras = []
    index_word = tokenizer_saida.index_word

    for idx in pred_indices:
        palavra = index_word.get(idx, "")
        if palavra == "<end>":
            break
        if palavra not in ["<start>", ""]:
            palavras.append(palavra)

    return " ".join(palavras)

print(traduzir("good morning"))
print(traduzir("thank you"))
print(traduzir("good night"))
print(traduzir("world"))

bom dia
obrigado
boa noite
olá


In [ ]:
print(traduzir("i love you"))

eu te amo amo


# 1.2 Transformer Tradutor



In [ ]:
!pip install transformers sentencepiece -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def traduzir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traduzir("We need your help Starfox!"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/779k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/799k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Necesitamos su ayuda Starfox!


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Modelo específico e robusto para tradução de Inglês para Português
model_name = "Helsinki-NLP/opus-mt-tc-big-en-pt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def traduzir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traduzir("So close, no matter how far"))
print(traduzir("Couldn't be much more from the heart"))
print(traduzir("Forever trusting who we are"))
print(traduzir("And nothing else matters"))
print(traduzir("Never opened myself this way"))
print(traduzir("Life is ours, we live it our way"))
print(traduzir("All these words, I don't just say"))
print(traduzir("And nothing else matters"))
print(traduzir("Trust I seek and I find in you"))
print(traduzir("Every day for us something new"))
print(traduzir("Open mind for a different view"))
print(traduzir("And nothing else matters"))
print(traduzir("Never cared for what they do"))
print(traduzir("Never cared for what they know"))
print(traduzir("But I know"))
print(traduzir("So close, no matter how far"))
print(traduzir("It couldn't be much more from the heart"))
print(traduzir("Forever trusting who we are"))
print(traduzir("And nothing else matters"))
print(traduzir("Never cared for what they do"))
print(traduzir("Never cared for what they know"))
print(traduzir("But I know"))
print(traduzir("I never opened myself this way"))
print(traduzir("Life is ours, we live it our way"))
print(traduzir("All these words, I don't just say"))
print(traduzir("And nothing else matters"))
print(traduzir("Trust I seek and I find in you"))
print(traduzir("Every day for us something new"))
print(traduzir("Open mind for a different view"))
print(traduzir("And nothing else matters"))
print(traduzir("Never cared for what they say"))
print(traduzir("Never cared for games they play"))
print(traduzir("Never cared for what they do"))
print(traduzir("Never cared for what they know"))
print(traduzir("And I know, yeah, yeah"))
print(traduzir("So close, no matter how far"))
print(traduzir("Couldn't be much more from the heart"))
print(traduzir("Forever trusting who we are"))
print(traduzir("No, nothing else matters"))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/825k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/465M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Tão perto, não importa o quão longe
Não poderia ser muito mais do coração
Para sempre confiando em quem somos
E nada mais importa
Nunca me abri assim
A vida é nossa, vivemos do nosso jeito
Todas essas palavras, eu não digo apenas
E nada mais importa
Eu procuro e encontro confiança em você
Todos os dias para nós algo novo
Mente aberta para uma visão diferente
E nada mais importa
Nunca se importou com o que eles fazem
Nunca se importou com o que eles sabem
Mas eu sei
Tão perto, não importa o quão longe
Não poderia ser muito mais do coração
Para sempre confiando em quem somos
E nada mais importa
Nunca se importou com o que eles fazem
Nunca se importou com o que eles sabem
Mas eu sei
Eu nunca me abri dessa maneira
A vida é nossa, vivemos do nosso jeito
Todas essas palavras, eu não digo apenas
E nada mais importa
Eu procuro e encontro confiança em você
Todos os dias para nós algo novo
Mente aberta para uma visão diferente
E nada mais importa
Nunca se importaram com o que dizem
Nunca se impo

A tradução preservou parcialmente o sentido original do trecho escolhido. Escolhi a música "Nothing Else Matters", da banda Metallica. De modo geral, a tradução conseguiu manter a mensagem principal da letra. No entanto, durante a aula eu não percebi, mas fazendo essa atividade, pude observar que alguns trechos ficaram literais demais, como a expressão "I don’t just say", traduzida como "eu não digo apenas". Embora essa tradução não esteja errada, ela soa pouco natural para nós,  onde o mais adequado seria "Eu não digo apenas isso" ou "Eu não digo isso da boca para fora!", pois preserva melhor o sentido da expressão no contexto da música. Logo, considero que a tradução manteve a ideia central, mas poderia ser aprimorada em alguns pontos para ficar mais natural e fiel a letra.

# ETAPA 2 — TF-IDF e análise sentimental com VADER

## 2.1 Modelo de classificação de termos com TF-IDF

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criando os documentos
documentos = [
    "O chatbot é inteligente e responde bem.",
    "O chatbot usa Machine Learning para aprender.",
    "Machine Learning é importante para IA."
]

# Criando o modelo TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documentos)

# Mostrando os termos e suas pontuações
print(vectorizer.get_feature_names_out())  # Lista de palavras analisadas
print(tfidf_matrix.toarray())  # Matriz TF-IDF dos documentos

['aprender' 'bem' 'chatbot' 'ia' 'importante' 'inteligente' 'learning'
 'machine' 'para' 'responde' 'usa']
[[0.         0.52863461 0.40204024 0.         0.         0.52863461
  0.         0.         0.         0.52863461 0.        ]
 [0.48148213 0.         0.36617957 0.         0.         0.
  0.36617957 0.36617957 0.36617957 0.         0.48148213]
 [0.         0.         0.         0.51741994 0.51741994 0.
  0.3935112  0.3935112  0.3935112  0.         0.        ]]


Artigo 1:

ESHUN, Michael; MENSAH, Rose-Mary Owusuaa; TAKYI, Kate; APPIAHENE, Peter; PEASAH, Kwame Ofosuhene; AMOAKO, Linda Banning. Sentiment analysis and classification of Ghanaian football tweets from the 2022 FIFA World Cup. Indonesian Journal of Electrical Engineering and Computer Science, v. 34, n. 1, p. 497-507, abr. 2024. DOI: 10.11591/ijeecs.v34.i1.pp497-507.


Artigo 2:

ALOUFI, Samah; EL SADDIK, Abdulmotaleb. Sentiment identification in football-specific tweets. IEEE Access, v. 6, p. 78609-78621, 2018. DOI: 10.1109/ACCESS.2018.2885117.

Algritmo da aula

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criando os documentos
documentos = [
    "Football as an attractive sport generates huge volumes of tweets concerning fans' opinions, feelings, and judgments during prime events. Such data can be leveraged in sentiment analysis, an algorithmic approach to analyzing text in tweets by extracting emotional tones. This paper presents a novel benchmark dataset of 132,115 tweets collected during the 2022 world cup on (formerly Twitter) for football-related sentiment classification. We also performed sentiment analysis on the dataset using lexicon-based tools, traditional machine learning algorithms, and pre-trained models, robustly optimized bidirectional encoder representations from transformers (BERT)pretraining approach RoBERTa and distilled version of BERT (DistilBERT) to understand the emotions and reactions of football fans during different phases of the football matches. Results from the study indicate that most tweets had neutral sentiments in both context-aware and context-free analysis. We also describe our novel GhaFootBERT, a sentiment classification model based on transfer learning on BERT, which provides an effective approach to sentiment classification of football-related tweets. Our model performs robustly, outperforming the traditional models with 92% accuracy.",
    "Sports fans generate a large amount of tweets which reflect their opinions and feelings about what is happening during various sporting events. Given the popularity of football events, in this work, we focus on analyzing sentiment expressed by football fans through Twitter. These tweets reflect the changes in the fans’ sentiment as they watch the game and react to the events of the game, e.g., goal scoring, penalties, and so on. Collecting and examining the sentiment conveyed through these tweets will help to draw a complete picture which expresses fan interaction during a specific football event. The objective of this work is to propose a domain-specific approach for understanding sentiments expressed in football fans’ conversations. To achieve our goal, we start by developing a football-specific sentiment dataset which we label manually. We then utilize our dataset to automatically create a football-specific sentiment lexicon. Finally, we develop a sentiment classifier which is capable of recognizing sentiments expressed in football  conversation. We conduct extensive experiments on our dataset to compare the performance of different learning algorithms in identifying the sentiment expressed in football related tweets. Our results show that our approach is effective in recognizing the fans’ sentiment during football events."
]

# Criando o modelo TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documentos)

# Mostrando os termos e suas pontuações
print(vectorizer.get_feature_names_out())  # Lista de palavras analisadas
print(tfidf_matrix.toarray())  # Matriz TF-IDF dos documentos

['115' '132' '2022' '92' 'about' 'accuracy' 'achieve' 'algorithmic'
 'algorithms' 'also' 'amount' 'an' 'analysis' 'analyzing' 'and' 'approach'
 'as' 'attractive' 'automatically' 'aware' 'based' 'be' 'benchmark' 'bert'
 'bidirectional' 'both' 'by' 'can' 'capable' 'changes' 'classification'
 'classifier' 'collected' 'collecting' 'compare' 'complete' 'concerning'
 'conduct' 'context' 'conversation' 'conversations' 'conveyed' 'create'
 'cup' 'data' 'dataset' 'describe' 'develop' 'developing' 'different'
 'distilbert' 'distilled' 'domain' 'draw' 'during' 'effective' 'emotional'
 'emotions' 'encoder' 'event' 'events' 'examining' 'experiments'
 'expressed' 'expresses' 'extensive' 'extracting' 'fan' 'fans' 'feelings'
 'finally' 'focus' 'football' 'for' 'formerly' 'free' 'from' 'game'
 'generate' 'generates' 'ghafootbert' 'given' 'goal' 'had' 'happening'
 'help' 'huge' 'identifying' 'in' 'indicate' 'interaction' 'is'
 'judgments' 'label' 'large' 'learning' 'leveraged' 'lexicon' 'machine'
 'manu

Algoritmo modificado
Criei essa versão para visualizar as informações deforma as adequada/didática, devido ao tamanho/quantidade de dados da tabela.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

documentos = [
    "Football as an attractive sport generates huge volumes of tweets concerning fans' opinions, feelings, and judgments during prime events. Such data can be leveraged in sentiment analysis, an algorithmic approach to analyzing text in tweets by extracting emotional tones. This paper presents a novel benchmark dataset of 132,115 tweets collected during the 2022 world cup on (formerly Twitter) for football-related sentiment classification. We also performed sentiment analysis on the dataset using lexicon-based tools, traditional machine learning algorithms, and pre-trained models, robustly optimized bidirectional encoder representations from transformers (BERT)pretraining approach RoBERTa and distilled version of BERT (DistilBERT) to understand the emotions and reactions of football fans during different phases of the football matches. Results from the study indicate that most tweets had neutral sentiments in both context-aware and context-free analysis. We also describe our novel GhaFootBERT, a sentiment classification model based on transfer learning on BERT, which provides an effective approach to sentiment classification of football-related tweets. Our model performs robustly, outperforming the traditional models with 92% accuracy.",
    "Sports fans generate a large amount of tweets which reflect their opinions and feelings about what is happening during various sporting events. Given the popularity of football events, in this work, we focus on analyzing sentiment expressed by football fans through Twitter. These tweets reflect the changes in the fans’ sentiment as they watch the game and react to the events of the game, e.g., goal scoring, penalties, and so on. Collecting and examining the sentiment conveyed through these tweets will help to draw a complete picture which expresses fan interaction during a specific football event. The objective of this work is to propose a domain-specific approach for understanding sentiments expressed in football fans’ conversations. To achieve our goal, we start by developing a football-specific sentiment dataset which we label manually. We then utilize our dataset to automatically create a football-specific sentiment lexicon. Finally, we develop a sentiment classifier which is capable of recognizing sentiments expressed in football conversation. We conduct extensive experiments on our dataset to compare the performance of different learning algorithms in identifying the sentiment expressed in football related tweets. Our results show that our approach is effective in recognizing the fans’ sentiment during football events."
]

# é pra romover os the, of, and, in...
vectorizer = TfidfVectorizer(stop_words='english')

# Aplicando o TF-IDF
tfidf_matrix = vectorizer.fit_transform(documentos)

# Vocabulário identificado
termos = vectorizer.get_feature_names_out()
print("Vocabulário identificado:")
print(termos)

# Matriz TF-IDF
matriz_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=termos,
    index=["Artigo 1", "Artigo 2"]
)

print("\nMatriz TF-IDF:")
print(matriz_tfidf)

# Termos mais relevantes de cada artigo
for i, artigo in enumerate(matriz_tfidf.index):
    print(f"\nTermos mais relevantes do {artigo}:")
    print(matriz_tfidf.loc[artigo].sort_values(ascending=False).head(10))

Vocabulário identificado:
['115' '132' '2022' '92' 'accuracy' 'achieve' 'algorithmic' 'algorithms'
 'analysis' 'analyzing' 'approach' 'attractive' 'automatically' 'aware'
 'based' 'benchmark' 'bert' 'bidirectional' 'capable' 'changes'
 'classification' 'classifier' 'collected' 'collecting' 'compare'
 'complete' 'concerning' 'conduct' 'context' 'conversation'
 'conversations' 'conveyed' 'create' 'cup' 'data' 'dataset' 'develop'
 'developing' 'different' 'distilbert' 'distilled' 'domain' 'draw'
 'effective' 'emotional' 'emotions' 'encoder' 'event' 'events' 'examining'
 'experiments' 'expressed' 'expresses' 'extensive' 'extracting' 'fan'
 'fans' 'feelings' 'finally' 'focus' 'football' 'free' 'game' 'generate'
 'generates' 'ghafootbert' 'given' 'goal' 'happening' 'help' 'huge'
 'identifying' 'indicate' 'interaction' 'judgments' 'label' 'large'
 'learning' 'leveraged' 'lexicon' 'machine' 'manually' 'matches' 'model'
 'models' 'neutral' 'novel' 'objective' 'opinions' 'optimized'
 'outperform

Os artigos compartilham como princiopais termos, "football", "sentiment", "tweets", "fans" e "dataset".
Analisando as tabelas de saida, o artigo 1 apresenta maior destaque para modelos computacionais avançados, falando sobre modelos ,  aboradgens, classificação... Já o  artigo  2 enfatiza expressões, torcedores, eventos e datasets. Com isso, pode-se notar que os artigos são bastatne similares, o que era esperado, mas já se pode notar um enfoque metodológico diferente.

## 2.2 Modelo de análise sentimental com VADER

Fiquei na duvida dessa parte, o enunciado fala em slides 34 e 40, ou seria 37 e 40?

Vídeo escolhido: https://www.youtube.com/watch?v=qIJrjTaJrbg

In [10]:

!pip install vaderSentiment

#IMPORTAÇÃO DAS BIBLIOTECAS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

#INICIALIZAÇÃO DO ANALISADOR DE SENTIMENTO
analisador = SentimentIntensityAnalyzer()

#DEFINIÇÃO DA FUNÇÃO DE ANÁLISE DE SENTIMENTO
def analisar_sentimento_vader(texto):

    #CÁLCULO DO SENTIMENTO
    sentimento = analisador.polarity_scores(texto)

    #CLASSIFICAÇÃO DO SENTIMENTO
    if sentimento['compound'] >= 0.05:
        return "Positivo"
    elif sentimento['compound'] <= -0.05:
        return "Negativo"
    else:
        return "Neutro"

# TESTE DA FUNÇÃO
mensagem1 = "O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida."
print(analisar_sentimento_vader(mensagem1))

mensagem2 = "Que baile do tricolor! Se termina-se 3 ou 4 a 0, no 1°T ,não seria injusto. Uma aula de futebol. E um baita time tinha o Grêmio... Título merecidíssimo..."
print(analisar_sentimento_vader(mensagem2))

mensagem3 = "olha o time do corintians que o gremio ganhou, monte de jogador bom. e o gremio tinha ido mal na primeira partida. esse gremio que a torcida sente saudades."
print(analisar_sentimento_vader(mensagem3))

mensagem4 = "Tempo q dava gosto de assistir futebol brasileiro"
print(analisar_sentimento_vader(mensagem4))

mensagem5 = "Sou Corinthians e assisti os 2 jogos. Esse 2° jogo o Grêmio jogou muito concentrado. Timaço!"
print(analisar_sentimento_vader(mensagem5))



Neutro
Negativo
Neutro
Neutro
Neutro


O modelo VADER classificou corretamente os comentários? Houve alguma limitação relacionada ao idioma, ironia, contexto ou tamanho do texto?

1 - O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida.

O VADER não classificou corretamente! Devido ao comentário usar termos como "acabou com a partida", o VADER foi muito literal. No sentido do comentário, seria algo positivo, pois querdizer que o jogador jogou muito bem.

2 - Que baile do tricolor! Se termina-se 3 ou 4 a 0, no 1°T ,não seria injusto. Uma aula de futebol. E um baita time tinha o Grêmio... Título merecidíssimo...

O VADER não classificou corretamente. Claramente o comentário é positivo.

3 - olha o time do corintians que o gremio ganhou, monte de jogador bom. e o gremio tinha ido mal na primeira partida. esse gremio que a torcida sente saudades.

O VADER classificou corretamente, apesar de ter uma dualidade. O início do comentário é positivo, elogiando os times e o jogo, mas o final é negativo, por destacar que o Grêmio jogou mal o primeiro jogo e devido ao sentimento de saudade devido ao time atual ser inferior em desempenho ao time destacado no vídeo.

4 - Tempo q dava gosto de assistir futebol brasileiro

Acredito que corretamente como NEUTRO. Pois elogia o futebol apresentado na época do vídeo, mas com isso, destaca que o futebol atual não possui o mesmo nível.

5 - Sou Corinthians e assisti os 2 jogos. Esse 2° jogo o Grêmio jogou muito concentrado. Timaço!

O VADER não classificou corretamente. O comentário é positivo, possui elogios ao time.

# ETAPA 3 — Análise sentimental com modelos Transformers
## 3.1 Modelo sentimental com RoBERTa

OBS: usei os mesmos comentários do vídeo do youtube para comparar os dois modelos!

In [7]:
from transformers import pipeline

# Criar pipeline de análise de sentimento multilíngue
analise_sentimento = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")

# Teste
mensagem1 = "O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida."
resultado1 = analise_sentimento(mensagem1)
print(resultado1)

mensagem2 = "Que baile do tricolor! Se termina-se 3 ou 4 a 0, no 1°T ,não seria injusto. Uma aula de futebol. E um baita time tinha o Grêmio... Título merecidíssimo..."
resultado2 = analise_sentimento(mensagem2)
print(resultado2)

mensagem3 = "olha o time do corintians que o gremio ganhou, monte de jogador bom. e o gremio tinha ido mal na primeira partida. esse gremio que a torcida sente saudades."
resultado3 = analise_sentimento(mensagem3)
print(resultado3)

mensagem4 = "Tempo q dava gosto de assistir futebol brasileiro"
resultado4 = analise_sentimento(mensagem4)
print(resultado4)

mensagem5 = "Sou Corinthians e assisti os 2 jogos. Esse 2° jogo o Grêmio jogou muito concentrado. Timaço!"
resultado5 = analise_sentimento(mensagem5)
print(resultado5)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'negative', 'score': 0.8252692222595215}]
[{'label': 'positive', 'score': 0.793099582195282}]
[{'label': 'negative', 'score': 0.4116162359714508}]
[{'label': 'neutral', 'score': 0.4572942852973938}]
[{'label': 'positive', 'score': 0.8459815979003906}]



1 - O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida.

O ROBERTA não classificou corretamente! Devido ao comentário usar termos como "acabou com a partida", o modelo foi muito literal. No sentido do comentário, seria algo positivo, pois querdizer que o jogador jogou muito bem.

2 - Que baile do tricolor! Se termina-se 3 ou 4 a 0, no 1°T ,não seria injusto. Uma aula de futebol. E um baita time tinha o Grêmio... Título merecidíssimo...

O ROBERTA classificou corretamente.

3 - olha o time do corintians que o gremio ganhou, monte de jogador bom. e o gremio tinha ido mal na primeira partida. esse gremio que a torcida sente saudades.

O ROBERTA não classificou corretamente, apesar de ter uma dualidade. O início do comentário é positivo, elogiando os times e o jogo, mas o final é negativo, por destacar que o Grêmio jogou mal o primeiro jogo e devido ao sentimento de saudade devido ao time atual ser inferior em desempenho ao time destacado no vídeo.

4 - Tempo q dava gosto de assistir futebol brasileiro

Acredito que corretamente como NEUTRO. Pois elogia o futebol apresentado na época do vídeo, mas com isso, destaca que o futebol atual não possui o mesmo nível.

5 - Sou Corinthians e assisti os 2 jogos. Esse 2° jogo o Grêmio jogou muito concentrado. Timaço!

O ROBERTA classificou corretamente.

## 3.2 BERT/BERTimbau

In [11]:
from transformers import pipeline

# Criar pipeline de análise de sentimento multilíngue
analise_sentimento = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

# Teste
mensagem1 = "O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida."
resultado1 = analise_sentimento(mensagem1)
print(resultado1)

mensagem2 = "Que baile do tricolor! Se termina-se 3 ou 4 a 0, no 1°T ,não seria injusto. Uma aula de futebol. E um baita time tinha o Grêmio... Título merecidíssimo..."
resultado2 = analise_sentimento(mensagem2)
print(resultado2)

mensagem3 = "olha o time do corintians que o gremio ganhou, monte de jogador bom. e o gremio tinha ido mal na primeira partida. esse gremio que a torcida sente saudades."
resultado3 = analise_sentimento(mensagem3)
print(resultado3)

mensagem4 = "Tempo q dava gosto de assistir futebol brasileiro"
resultado4 = analise_sentimento(mensagem4)
print(resultado4)

mensagem5 = "Sou Corinthians e assisti os 2 jogos. Esse 2° jogo o Grêmio jogou muito concentrado. Timaço!"
resultado5 = analise_sentimento(mensagem5)
print(resultado5)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': '1 star', 'score': 0.4164504408836365}]
[{'label': '5 stars', 'score': 0.6526293158531189}]
[{'label': '5 stars', 'score': 0.385026752948761}]
[{'label': '4 stars', 'score': 0.3285106420516968}]
[{'label': '1 star', 'score': 0.2992643713951111}]



1 - O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida.

O BERTimbau não classificou corretamente! Devido ao comentário usar termos como "acabou com a partida", o modelo foi muito literal. No sentido do comentário, seria algo positivo, pois quer dizer que o jogador jogou muito bem. Exatamente como os demais modelos.

2 - Que baile do tricolor! Se termina-se 3 ou 4 a 0, no 1°T ,não seria injusto. Uma aula de futebol. E um baita time tinha o Grêmio... Título merecidíssimo...

O BERTimbau classificou corretamente.

3 - olha o time do corintians que o gremio ganhou, monte de jogador bom. e o gremio tinha ido mal na primeira partida. esse gremio que a torcida sente saudades.

O BERTimbau classificou corretamente, apesar de ter uma dualidade. O início do comentário é positivo, elogiando os times e o jogo, mas o final é negativo, por destacar que o Grêmio jogou mal o primeiro jogo e devido ao sentimento de saudade devido ao time atual ser inferior em desempenho ao time destacado no vídeo. Devido a isso, apesar das 5 estrelas tal qual como o comentário anterior, o score foi bem inferior!

4 - Tempo q dava gosto de assistir futebol brasileiro

Acredito que corretamente como um "positivo fraco". Pois elogia o futebol apresentado na época do vídeo comparado ao futebol atual.

5 - Sou Corinthians e assisti os 2 jogos. Esse 2° jogo o Grêmio jogou muito concentrado. Timaço!

O BERTimbau classificou como negativo, o que pode parecer estranho. Talvez a interpretação do modelo levou em consideração que o autor do comentário é corintiano, e elogiu o time adversário.

## Comparativo

Comparando os dois modelos de análise de sentimentos, observa-se que ambos apresentaram bons resultados em comentários mais claramente positivos, como nos casos em que aparecem expressões como "uma aula de futebol", "título merecidíssimo" e "timaço". Nesses exemplos, tanto o modelo RoBERTa quanto o modelo BERTimbau  conseguiram identificar corretamente o tom positivo dos comentários.

Porém, os modelos apresentaram dificuldades em comentários com linguagem mais informal. Um exemplo foi a frase "O que o Zinho fez nesse jogo foi brincadeira. Jogou de terno e acabou com a partida". Apesar de o sentido real do comentário ser positivo, pois indica que o jogador teve uma grande atuação, os modelos interpretaram de forma mais literal expressões como "acabou com a partida", o que prejudicou a classificação correta do sentimento.

Também foi possível perceber diferenças entre os modelos em comentários com maior ambiguidade, como no terceiro comentário, onde há uma mistura de elogio ao time antigo do Grêmio com uma comparação implícita ao momento atual, marcada pelo sentimento de saudade. O RoBERTa teve mais dificuldade em lidar com essa dualidade, enquanto o BERTimbau se saiu melhor, com uma classificação mais adequada, ainda que com menor confiança em relação a comentários mais diretamente positivos.

O quarto comentário, "Tempo q dava gosto de assistir futebol brasileiro", pode ser interpretada como positiva, por elogiar o futebol de uma época, mas também carrega uma crítica ao futebol atual. Por isso, classificações como neutra ou positivo fraco podem ser consideradas aceitáveis, dependendo da leitura do modelo.

Assim, concluo que os dois algoritmos são úteis para análise automática de sentimentos, mas seus resultados precisam ser interpretados criticamente. A análise humana continua sendo importante para verificar casos em que o sentido real do comentário não está apenas nas palavras utilizadas, mas no contexto em que elas aparecem.